# IRIS V3 Conceptual Report
This document details the internal logic and algorithms of the IRIS V3 VST plugin to serve as a reference for reimplementation in Unreal Engine.

## 1. Coordinate System & World State
The virtual acoustic space is a 2D plane normalized to a unit square.

Bounds: [0.0, 1.0] on both X and Y axes.
Listener: Represented as a point  (x, y) within these bounds.
IR Sources: Represented as points (x, y) associated with a convolution engine.
Walls: Represented as line segments from (x1, y1) to (x2, y2).

All distance calculations and logic use this normalized coordinate space.

## 2. Listener Logic
The listener's position is the primary driver for all DSP updates.

### 2.1. Movement & Inertia
The listener moves towards a target position (set by mouse or OSC) with simulated inertia to smooth out jumps and rapid movements.

Update Rate: 60 Hz (via timerCallback).
Inertia Factor: Controlled by the Inertia parameter (0.0 = instant, 1.0 = slow).
Algorithm: cpp // simple linear interpolation towards target
float alpha = 1.0f - 0.98f * inertiaParam; 
currentPos += (targetPos - currentPos) * alpha;
Freeze: If Freeze is enabled, the listener position ignores updates.

### 2.2. Wall Constraints (Collision)
The listener is prevented from passing through walls.

Clearance: 0.02 units (normalized).
Logic: For every wall, the closest point on the wall segment to the listener is calculated. If the distance d is less than 0.02:
Calculate a push vector away from the wall (along the normal).
Push the listener so distance becomes 0.02.
This effectively makes walls "solid" with a small buffer.

## 3. Wall Computation (Occlusion & Diffraction)
Walls act as occluders that modify the weight (gain) of IR sources.

### 3.1. Occlusion Check
For every IR source, a line of sight (ray) is cast from the Listener to the IR.

Intersection: The system checks if this ray intersects any Wall segment.
Attenuation:
- Each wall has an attenuation factor (derived from Wall Opacity parameter).
- Base Opacity = Wall Opacity parameter.
- If an intersection occurs, the IR's gain is multiplied by (1.0 - EffectiveOpacity).
Multiple walls stack multiplicatively.

### 3.2. Edge Diffraction (Edge Fade)
Walls are not perfectly sharp; they simulate diffraction at the edges (tips) to allow sound to wrap around slightly.

Intersection Point (t): Calculated as a normalized position along the wall segment (0.0 at one end, 1.0 at the other).
Fade Zone: The first and last 20% of the wall length.
Fade Factor:
- If t < 0.2: factor = t / 0.2 (Linear fade from 0% opacity at tip to 100% base opacity).
- If t > 0.8: factor = (1.0 - t) / 0.2 (Linear fade out).
Otherwise: factor = 1.0 (Full opacity).
Effective Opacity: Base Opacity * Fade Factor.
Result: Sound passing near the edge of a wall is less attenuated than sound hitting the center.

## 4. IR Selection & Weighting
The core DSP logic determines which IRs are active and their respective gains (weights).

### 4.1. Gaussian Weighting Field
Every IR generates a Gaussian field of influence.

Formula: $$w_{raw} = e^{-\frac{d^2}{2\sigma^2}}$$ Where $d$ is the distance between Listener and IR.
Spread ($\sigma$): Controlled by Spread parameter. $$\sigma = 0.05 + 1.5 \times spread^2$$ (Range approx 0.05 to 1.55).

### 4.2. Active Set Selection (Hysteresis)
To save CPU, only a subset of IRs are convolved (Active Set).

Sorting: All IRs are sorted by their calculate raw_weight (after occlusion).
Rule A (Minimum Set): The top 4 IRs are always active.
Rule B (Dynamic Set): Up to 8 IRs can be active. Additional IRs (rank 5-8) are included if their weight exceeds a threshold relative to the loudest IR ($w_{max}$).
Entry Threshold: $w > 0.1 \times w_{max}$.
Exit Threshold: $w < 0.05 \times w_{max}$. This hysteresis prevents rapid toggling of convolvers at the edge of the field.

### 4.3. Weight Smoothing & Normalization
Smoothing: The target weights for all IRs are smoothed using a one-pole filter ($\alpha \approx 0.25$ per frame at 60Hz) to prevent clicks during movement.
Inactive IRs: IRs not in the active set fade their smooth weight to 0.
Normalization: The smoothed weights of all active IRs are summed (TotalWeight). DSP Weights are normalized so they sum to 1.0 relative to each other: $$w_{dsp} = \frac{w_{smooth}}{TotalWeight}$$

## 5. Audio Processing Engine
### 5.1. Convolution
Engine: juce::dsp::Convolution.
Process: Mono Input is fed to all Active Convolvers. Each Convolver output is scaled by its normalized $w_{dsp}$.
Outputs are summed to a wet buffer.

### 5.2. Dry/Wet Mix (Dynamic)
The mix logic adapts to the listener's position in the Gaussian field.

User Mix: The Mix knob value.
Field Strength: TotalWeight (sum of un-normalized smoothed weights).
Effective Wet: $$WetGain = UserMix \times \min(1.0, TotalWeight)$$ If the listener is far from all IRs (TotalWeight < 1.0), the wet signal fades out naturally.
Dry Gain: $$DryGain = 1.0 - WetGain$$ Note: This is a linear crossfade logic applied to the final output.

## 6. Implementation Summary for Unreal Engine
To recreate this in Unreal:

Spatial Query:

Every tick, calculate distance $d$ from Listener to all IR Actors.
Calculate Gaussian weight.
Perform Line Trace (Raycast) from Listener to IR to detect Walls.
Apply Wall attenuation (consider implementing edge fade logic in the wall material or math).
Selection Logic:

Sort results by weight.
Pick top 4 + up to 4 more based on threshold.
Manage a pool of Submix Effects (Convolution) or Metasounds.
Smoothing:

Interpolate the gains (weights) of the active convolutions over time (e.g., FInterpTo).
Normalization:

Ensure the gains sent to the convolution engines verify $\sum gain = 1.0$ (scaled by global mix).
Listener Constraints: Use UE's built-in collision/physics for the listener (Pawn) against Wall actors (Static Mesh with collision). The custom "push" logic in IRIS V3 is essentially a simple circle-capsule collision solver, which UE physics handles natively.